In [3]:
# 1. 필요한 라이브러리 불러오기
import pandas as pd
import os

# 2. 그룹 정의하기 (수정된 부분)
# '콜백스미싱' -> '콜백스미싱형'으로 수정했습니다.
# 만약 다른 유형도 이름이 다르다면, 위 진단 코드 결과를 보고 맞춰주세요.
groups_to_create = {
    "스미싱_세금_투자형": ["콜백스미싱형", "세금환급형", "투자권유형"],
    "기관사칭형": ["기관사칭형"],
    "메신저_택배_가족지인형": ["메신저피싱형", "택배사칭형", "가족지인사칭형"],
    "대출빙자형": ["대출빙자형"],
}

# 3. 원본 CSV 파일 불러오기
input_filename = "phishing_total_2046.csv"
df = None

try:
    df = pd.read_csv(input_filename)
    print(f"'{input_filename}' 파일을 성공적으로 불러왔습니다. (총 {len(df)}개 행)")

    # 만약을 위해, 각 유형 이름의 앞뒤 공백을 제거합니다. (코드 안정성 강화)
    df["phishing_type"] = df["phishing_type"].str.strip()

except FileNotFoundError:
    print(f"오류: '{input_filename}' 파일을 찾을 수 없습니다.")

# 4. 데이터프레임이 성공적으로 로드되었을 때만 다음 작업 수행
if df is not None:
    # 결과를 저장할 폴더 생성
    output_folder = "output_by_custom_group"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"'{output_folder}' 폴더를 생성했습니다.")

    print("\n지정된 그룹별로 파일 분리를 시작합니다...")

    # 5. 정의된 각 그룹에 대해 반복 작업
    for group_name, phishing_types in groups_to_create.items():
        group_df = df[df["phishing_type"].isin(phishing_types)]

        # 그룹에 데이터가 하나라도 있을 경우에만 파일을 생성합니다.
        if not group_df.empty:
            output_filename = os.path.join(output_folder, f"grouped_{group_name}.csv")
            group_df.to_csv(output_filename, index=False, encoding="utf-8-sig")
            print(
                f"- '{group_name}' 그룹 ({len(group_df)}개 행) -> '{output_filename}' 파일로 저장 완료."
            )
        else:
            print(
                f"- '{group_name}' 그룹에 해당하는 데이터가 없습니다. 파일을 생성하지 않습니다."
            )

    print("\n모든 파일 분리 작업이 완료되었습니다.")
else:
    print("파일을 읽지 못했기 때문에, 파일 분리 작업을 진행할 수 없습니다.")

'phishing_total_2046.csv' 파일을 성공적으로 불러왔습니다. (총 44519개 행)
'output_by_custom_group' 폴더를 생성했습니다.

지정된 그룹별로 파일 분리를 시작합니다...
- '스미싱_세금_투자형' 그룹 (2491개 행) -> 'output_by_custom_group\grouped_스미싱_세금_투자형.csv' 파일로 저장 완료.
- '기관사칭형' 그룹 (27823개 행) -> 'output_by_custom_group\grouped_기관사칭형.csv' 파일로 저장 완료.
- '메신저_택배_가족지인형' 그룹 (3471개 행) -> 'output_by_custom_group\grouped_메신저_택배_가족지인형.csv' 파일로 저장 완료.
- '대출빙자형' 그룹 (10734개 행) -> 'output_by_custom_group\grouped_대출빙자형.csv' 파일로 저장 완료.

모든 파일 분리 작업이 완료되었습니다.


In [2]:
# 1. 필요한 라이브러리 불러오기
import os
import pandas as pd
import openai
from dotenv import load_dotenv
from tqdm.notebook import tqdm

# 2. .env 파일에서 환경 변수 불러오기
load_dotenv()

# 3. OpenAI 클라이언트 생성
# (이전 코드와 동일)
try:
    api_key = os.getenv("GPT_OPENAI_API")
    if api_key is None:
        raise ValueError("환경 변수 'GPT_OPENAI_API'를 찾을 수 없습니다.")
    client = openai.OpenAI(api_key=api_key)
    print("인증 성공! OpenAI API를 사용할 수 있습니다.")
except Exception as e:
    print(f"클라이언트 생성 실패: {e}")
    exit()

# --- (수정된 부분) 실험을 위한 설정 변수 ---
test_config = {
    "intermediate_language": "English",  # "English", "Japanese", "Chinese" 등 변경 가능
    "temperature": 0.9,  # 0.7에서 0.9 또는 1.0으로 값을 높임
    "model": "gpt-4o-mini",
}
# ---------------------------------------------

# 4. 프롬프트 템플릿 정의
prompt_template = """
# PERSONA
You are an expert linguist specializing in data augmentation for an AI model. Your task is to paraphrase Korean sentences through back-translation while preserving their original core meaning and nuance.

# CONTEXT
The sentences are from voice phishing dialogues. It is important to maintain a natural, conversational tone. Some sentences might be urgent, persuasive, or slightly awkward, and this nuance should not be completely lost.

# TASK
Perform a back-translation on the [Korean Sentence] provided below by following these steps:
1.  Translate the [Korean Sentence] into {intermediate_language}.
2.  Translate the result from step 1 back into Korean.
3.  Output ONLY the final, back-translated Korean sentence without any additional text, explanations, or labels.

# INPUT
intermediate_language: {intermediate_language}
Korean Sentence: "{text}"
"""

# 5. CSV 파일 불러오기
input_filename = "jinjoo_test.csv"  # 테스트할 원본 파일
df = None
try:
    df = pd.read_csv(input_filename)
    print(f"'{input_filename}' 파일에서 {len(df)}개 행을 불러왔습니다.")
except FileNotFoundError:
    print(f"오류: '{input_filename}' 파일이 존재하지 않습니다.")
    exit()


# 6. 역번역 실행
augmented_texts = []
print(
    f"\nOpenAI API로 역번역을 시작합니다... (설정: 중간언어={test_config['intermediate_language']}, 온도={test_config['temperature']})"
)

# 원본 text를 보존하기 위해 복사본 사용
texts_to_translate = df["text"].copy()

for text in tqdm(texts_to_translate):
    try:
        prompt = prompt_template.format(
            intermediate_language=test_config["intermediate_language"], text=text
        )

        response = client.chat.completions.create(
            model=test_config["model"],
            messages=[{"role": "user", "content": prompt}],
            temperature=test_config["temperature"],
        )
        back_translated_text = response.choices[0].message.content.strip()
        augmented_texts.append(back_translated_text)

    except Exception as e:
        print(f"오류 발생: {e}, 원본 텍스트를 사용합니다: {text}")
        augmented_texts.append(text)

# 7. 결과 저장
# 원본과 비교하기 쉽도록, 이번에는 'augmented_text' 컬럼을 새로 추가해서 저장합니다.
df["augmented_text"] = augmented_texts
output_filename = f"final_augmented_daechul_varied.csv"
df.to_csv(output_filename, index=False, encoding="utf-8-sig")
print(f"\n작업 완료! 결과가 '{output_filename}' 파일로 저장되었습니다.")

# 8. 최종 결과 확인 (원본과 증강된 텍스트 비교)
print("\n--- 최종 작업 결과 샘플 ---")
print(df[["text", "augmented_text"]].head())

인증 성공! OpenAI API를 사용할 수 있습니다.
'jinjoo_test.csv' 파일에서 52개 행을 불러왔습니다.

OpenAI API로 역번역을 시작합니다... (설정: 중간언어=English, 온도=0.9)


  0%|          | 0/52 [00:00<?, ?it/s]


작업 완료! 결과가 'final_augmented_daechul_varied.csv' 파일로 저장되었습니다.

--- 최종 작업 결과 샘플 ---
                                            text  \
0            안녕하세요, OOO님 맞으신가요? 대출 관련해서 연락드렸습니다.   
1                                    네, 무슨 일인가요?   
2     저희 은행에서 특별 대출 상품을 제공하고 있습니다. 잠시 설명드려도 될까요?   
3  안녕하세요, OOO님 맞으신가요? 고객님께 좋은 대출 상품이 있어 연락드렸습니다.   
4                                대출이요? 어떤 상품인가요?   

                                    augmented_text  
0            안녕하세요, OOO님 맞으신가요? 저는 대출에 대해 연락드렸습니다.  
1                                    "네, 무슨 일이에요?"  
2  "저희 은행에서 특별한 대출 상품을 제공하고 있습니다. 간단히 설명해도 괜찮을까요?"  
3  "안녕하세요, OOO님 맞으신가요? 좋은 대출 상품이 있어 고객님께 연락드렸습니다."  
4              "Loan? What kind of product is it?"  


In [6]:
# 1. 필요한 라이브러리 불러오기
import os
import pandas as pd
import openai
from dotenv import load_dotenv
from tqdm.notebook import tqdm

# 2. .env 파일에서 환경 변수 불러오기
load_dotenv()

# 3. OpenAI 클라이언트 생성
try:
    api_key = os.getenv("GPT_OPENAI_API")
    if api_key is None:
        raise ValueError("환경 변수 'GPT_OPENAI_API'를 찾을 수 없습니다.")
    client = openai.OpenAI(api_key=api_key)
    print("인증 성공! OpenAI API를 사용할 수 있습니다.")
except Exception as e:
    print(f"클라이언트 생성 실패: {e}")
    exit()

# --- 실험을 위한 설정 변수 ---
test_config = {
    "intermediate_language": "Japanese",
    "temperature": 0.8,
    "model": "gpt-4o-mini",
}
# ---------------------------------------------

# 4. 프롬프트 템플릿 정의 (수정된 부분)
# 출력 규칙을 더 명확하고 강력하게 수정했습니다.
prompt_template = """
# PERSONA
You are an expert linguist specializing in data augmentation.

# TASK
Your task is to perform a back-translation of the given [Korean Sentence].
1.  Internally, translate the [Korean Sentence] into {intermediate_language}.
2.  Internally, translate the result from step 1 back into Korean.
3.  Provide the final, back-translated Korean sentence as the output.

# OUTPUT RULES (Strictly follow)
- Your response MUST be a single Korean sentence.
- Your response MUST NOT contain any English words, phrases, or characters.
- Your response MUST NOT include any labels like "Korean Sentence:" or explanations.
- Your response MUST be only the final Korean translation, nothing else.

# INPUT
intermediate_language: {intermediate_language}
Korean Sentence: "{text}"
"""

# 5. CSV 파일 불러오기
input_filename = "jinjoo_test.csv"
df = None
try:
    df = pd.read_csv(input_filename)
    print(f"'{input_filename}' 파일에서 {len(df)}개 행을 불러왔습니다.")
except FileNotFoundError:
    print(f"오류: '{input_filename}' 파일이 존재하지 않습니다.")
    exit()


# 6. 역번역 실행
augmented_texts = []
print(
    f"\nOpenAI API로 역번역을 시작합니다... (설정: 중간언어={test_config['intermediate_language']}, 온도={test_config['temperature']})"
)

for text in tqdm(df["text"]):
    try:
        prompt = prompt_template.format(
            intermediate_language=test_config["intermediate_language"], text=text
        )

        response = client.chat.completions.create(
            model=test_config["model"],
            messages=[{"role": "user", "content": prompt}],
            temperature=test_config["temperature"],
        )
        back_translated_text = response.choices[0].message.content.strip()
        augmented_texts.append(back_translated_text)

    except Exception as e:
        print(f"오류 발생: {e}, 원본 텍스트를 사용합니다: {text}")
        augmented_texts.append(text)

# 7. 결과 저장
df["text"] = augmented_texts
output_filename = f"final_augmented_daechul_korean_only.csv"
df.to_csv(output_filename, index=False, encoding="utf-8-sig")
print(f"\n작업 완료! 결과가 '{output_filename}' 파일로 저장되었습니다.")

# 8. 최종 결과 확인
print("\n--- 최종 작업 결과 샘플 ---")
print(df.head())

인증 성공! OpenAI API를 사용할 수 있습니다.
'jinjoo_test.csv' 파일에서 35개 행을 불러왔습니다.

OpenAI API로 역번역을 시작합니다... (설정: 중간언어=Japanese, 온도=0.8)


  0%|          | 0/35 [00:00<?, ?it/s]


작업 완료! 결과가 'final_augmented_daechul_korean_only.csv' 파일로 저장되었습니다.

--- 최종 작업 결과 샘플 ---
      file_name phishing_type  speaker  \
0  phishing_386         대출빙자형        0   
1  phishing_386         대출빙자형        1   
2  phishing_386         대출빙자형        0   
3  phishing_387         대출빙자형        0   
4  phishing_387         대출빙자형        1   

                                             text  
0            안녕하세요, OOO님 맞으신가요? 대출과 관련하여 연락드렸습니다.  
1                                     네, 무슨 일이에요?  
2  "저희 은행에서 특별한 대출 상품을 제공하고 있습니다. 잠깐 설명해도 괜찮을까요?"  
3  안녕하세요, OOO님 맞으신가요? 고객님께 좋은 대출 상품이 있어서 연락드렸습니다.  
4                               "대출이요? 어떤 종류인가요?"  


In [9]:
# 1. 필요한 라이브러리 불러오기
import os
import pandas as pd
import time
import re
from dotenv import load_dotenv
from openai import OpenAI

# .env에서 OpenAI API 키 불러오기
load_dotenv()

# --- (수정 필요 1: .env 파일의 API 키 이름 확인) ---
# 네가 .env 파일에 키를 저장할 때 사용한 이름으로 수정해야 해.
# 이전에 "GPT_OPENAI_API"로 저장했다면, 아래 변수 이름과 오류 메시지를 맞춰주는 게 좋아.
GPT_OPENAI_API = os.getenv("GPT_OPENAI_API")
if not GPT_OPENAI_API:
    raise ValueError(" .env에 GPT_OPENAI_API가 설정되지 않았습니다.")
# ----------------------------------------------------

client = OpenAI(api_key=GPT_OPENAI_API)


# 역번역 함수 (GPT 응답 줄 수 일치 확인 + 로그 저장)
def back_translate_with_openai(speaker_list, text_list, file_name):
    tagged_lines = [
        f"SPEAKER_{spk}: {txt}" for spk, txt in zip(speaker_list, text_list)
    ]
    joined_text = "\n".join(tagged_lines)

    prompt = f"""
한국어로 된 보이스피싱 대화를 역번역해서 데이터 증강. 각 문장은 SPEAKER_0: 또는 SPEAKER_1: 으로 시작.

다음 지침을 엄격히 따르세요.
1. 각 발화를 영어로 번역한 뒤, 다시 한국어로 번역.
2. 의미는 비슷하게 유지하되 표현은 다양하게, 유의어로 치환해도 됨.
3. **출력 형식은 반드시 SPEAKER 태그 + 원문과 동일한 줄 수(총 {len(text_list)}줄)를 유지.**
4. 줄 수가 다를 경우 작업은 실패.
5. **최종 출력은 반드시 한국어로만 출력합니다.**

입력:
{joined_text}

역번역 한국어 결과:
""".strip()

    try:
        # --- (수정 필요 2: 모델 및 번역 품질 설정 - 선택 사항) ---
        # 모델을 바꾸거나(예: "gpt-4o"), 번역 결과물의 다양성을 조절하고 싶을 때 수정.
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.8,
        )
        # ---------------------------------------------------------
        reply = response.choices[0].message.content

        # GPT 응답 저장
        with open("gpt_raw_log.txt", "a", encoding="utf-8") as log_file:
            log_file.write(f"\n\n===== {file_name} =====\n{reply}\n")

        # SPEAKER 태그로 분리
        matches = re.findall(r"(?i)SPEAKER[_ ]?(\d):\s*(.+)", reply)
        if len(matches) != len(text_list):
            print(
                f"[경고] 문장 수 불일치 - 원문: {len(text_list)}, GPT 응답: {len(matches)}"
            )
            # 실패 로그 저장
            with open("failed_files.txt", "a", encoding="utf-8") as fail_log:
                fail_log.write(file_name + "\n")
            return [None] * len(text_list)

        return [text for _, text in matches]

    except Exception as e:
        print(f"[OpenAI API 오류] {e}")
        with open("failed_files.txt", "a", encoding="utf-8") as fail_log:
            fail_log.write(file_name + "\n")
        return [None] * len(text_list)


# --- (수정 필요 3: 작업 대상 파일 및 유형 설정 - 가장 중요) ---
# 이 스크립트를 실행할 때마다 네가 작업하고 싶은 내용에 맞춰 이 부분을 바꿔주면 돼.

# 입력 파일 경로
INPUT_FILE = "jinjoo_test.csv"

# 결과가 저장될 파일 이름
OUTPUT_FILE = "jj_jgang.csv"

# 위 입력 파일에서, 실제로 역번역을 실행할 phishing_type 목록
TARGET_TYPES = ["대출빙자형"]
# -------------------------------------------------------------


# file_name 불러오기
def load_processed_file_names():
    if not os.path.exists(OUTPUT_FILE):
        return set()
    try:
        processed_df = pd.read_csv(OUTPUT_FILE)
        return set(processed_df["file_name"].unique())
    except:
        return set()


# 데이터 불러오기 및 필터
df = pd.read_csv(INPUT_FILE)
df = df[df["phishing_type"].isin(TARGET_TYPES)]
grouped = df.groupby("file_name")
processed_files = load_processed_file_names()

# CSV 실시간 저장
with open(OUTPUT_FILE, "a", encoding="utf-8", newline="") as f_out:
    # 파일이 비어있을 경우에만 헤더를 씁니다.
    if os.path.getsize(OUTPUT_FILE) == 0:
        f_out.write("file_name,speaker,phishing_type,text,back_translated\n")

    for file_name, group in grouped:
        if file_name in processed_files:
            continue  # 이미 처리된 대화는 건너뜀

        speaker_list = group["speaker"].tolist()
        text_list = group["text"].tolist()
        phishing_type = group["phishing_type"].iloc[0]

        backtranslated_list = back_translate_with_openai(
            speaker_list, text_list, file_name
        )

        for spk, orig, back in zip(speaker_list, text_list, backtranslated_list):
            # CSV 저장 시 쉼표나 개행문자로 인한 오류 방지
            orig_safe = str(orig).replace("\n", " ").replace(",", " ")
            back_safe = str(back or "").replace("\n", " ").replace(",", " ")
            f_out.write(f"{file_name},{spk},{phishing_type},{orig_safe},{back_safe}\n")

        print(f" 처리 완료: {file_name}")
        time.sleep(1.5)  # API 과부하 방지를 위한 대기 시간

 처리 완료: phishing_386
 처리 완료: phishing_387
 처리 완료: phishing_388
 처리 완료: phishing_389
 처리 완료: phishing_390
 처리 완료: phishing_391
 처리 완료: phishing_392
 처리 완료: phishing_393


KeyboardInterrupt: 

In [11]:
# 1. 필요한 라이브러리 불러오기
import os
import pandas as pd
import openai
from dotenv import load_dotenv
from tqdm.notebook import tqdm

# 2. .env 파일에서 환경 변수 불러오기
load_dotenv()

# 3. OpenAI 클라이언트 생성
try:
    api_key = os.getenv("GPT_OPENAI_API")
    if api_key is None:
        raise ValueError("환경 변수 'GPT_OPENAI_API'를 찾을 수 없습니다.")
    client = openai.OpenAI(api_key=api_key)
    print("인증 성공! OpenAI API를 사용할 수 있습니다.")
except Exception as e:
    print(f"클라이언트 생성 실패: {e}")
    exit()

# --- (수정 필요 1: 모델 및 창의성 설정) ---
# 더 좋은 품질을 위해 모델을 gpt-4o로 변경하고, temperature를 높이는 것을 추천합니다.
test_config = {
    "intermediate_language": "English",
    "temperature": 0.9,  # 창의성을 높이기 위해 0.7 -> 0.9로 상향
    "model": "gpt-4o-mini",  # gpt-4o-mini.
}


# --- (프롬프트 템플릿) ---
prompt_template = """
# PERSONA
You are a creative copywriter and data augmentation expert.

# PRIMARY GOAL
Your main goal is to rephrase a Korean sentence by performing back-translation.
The final output MUST be a new sentence that is semantically identical to the original but syntactically or lexically different.

# DETAILED INSTRUCTIONS
1.  **Paraphrase Actively**: Do not just translate back and forth. Actively use synonyms, change sentence structures, and adjust formality.
2.  **Preserve Nuance and Intent**: Maintain the original's core intent. A neutral question must remain neutral. A persuasive tone must remain persuasive. DO NOT change the fundamental meaning or the speaker's intention.
3.  **Strictly Korean Output**: The final output MUST be a single, complete Korean sentence and nothing else.

# EXAMPLES OF GOOD AUGMENTATION (What to do)
-   Original: "지금 바로 입금하지 않으시면 대출 승인이 취소될 수 있습니다."
-   Augmented: "즉시 송금하지 않을 경우, 해당 대출 건은 무효 처리될 수 있습니다."
-   Original: "아빠, 나 핸드폰 액정이 깨져서 친구 폰으로 문자했어. 급하게 돈 좀 보내줘."
-   Augmented: "아빠, 제 휴대폰 화면이 파손돼서 친구 전화로 연락드렸어요. 급히 이체해야 할 돈이 있어서요."

# EXAMPLE OF BAD AUGMENTATION (What NOT to do)
-   Original: "네, 무슨 일인가요?" (Correct: Neutral inquiry)
-   Augmented: "네, 어떤 문제가 발생한 건가요?" (Incorrect: Assumes a problem, changes intent)

# YOUR TASK
Now, apply these instructions to the following sentence.

Korean Sentence: "{text}"
"""

# 5. CSV 파일 불러오기
input_filename = "jinjoo_test.csv"
df = None
try:
    df = pd.read_csv(input_filename)
    print(f"'{input_filename}' 파일에서 {len(df)}개 행을 불러왔습니다.")
except FileNotFoundError:
    print(f"오류: '{input_filename}' 파일이 존재하지 않습니다.")
    exit()


# 6. 역번역 실행
augmented_texts = []
print(
    f"\nOpenAI API로 역번역을 시작합니다... (모델={test_config['model']}, 온도={test_config['temperature']})"
)

for text in tqdm(df["text"]):
    try:
        prompt = prompt_template.format(
            intermediate_language=test_config["intermediate_language"], text=text
        )

        response = client.chat.completions.create(
            model=test_config["model"],
            messages=[{"role": "user", "content": prompt}],
            temperature=test_config["temperature"],
        )
        back_translated_text = response.choices[0].message.content.strip()
        augmented_texts.append(back_translated_text)

    except Exception as e:
        print(f"오류 발생: {e}, 원본 텍스트를 사용합니다: {text}")
        augmented_texts.append(text)

# 7. 결과 저장
df["text"] = augmented_texts
output_filename = f"final_augmented_high_quality.csv"
df.to_csv(output_filename, index=False, encoding="utf-8-sig")
print(f"\n작업 완료! 결과가 '{output_filename}' 파일로 저장되었습니다.")

# 8. 최종 결과 확인
print("\n--- 최종 작업 결과 샘플 ---")
print(df.head())

인증 성공! OpenAI API를 사용할 수 있습니다.
'jinjoo_test.csv' 파일에서 11개 행을 불러왔습니다.

OpenAI API로 역번역을 시작합니다... (모델=gpt-4o-mini, 온도=0.9)


  0%|          | 0/11 [00:00<?, ?it/s]


작업 완료! 결과가 'final_augmented_high_quality.csv' 파일로 저장되었습니다.

--- 최종 작업 결과 샘플 ---
      file_name phishing_type  speaker  \
0  phishing_389         대출빙자형        0   
1  phishing_389         대출빙자형        1   
2  phishing_389         대출빙자형        0   
3  phishing_390         대출빙자형        0   
4  phishing_390         대출빙자형        1   

                                                text  
0   "OOO님, 안녕하세요. 저희 금융기관에서는 특별한 대출 혜택을 알려드리려고 합니다."  
1                          "특별한 대출의 경우, 어떤 요건이 있나요?"  
2  "이 조건은 고객님께서 상당한 이익을 누리실 수 있는 기회를 제공합니다. 들어보실래요?"  
3                  "OOO님이신가요? 대출 관련 상담을 위해 연락드렸습니다."  
4                            "저는 대출에 대해 별로 흥미가 없어요."  


In [4]:
# 1. 필요한 라이브러리 불러오기
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm
import time
import csv
import re

# --- (사용자 설정) ---
# 이 스크립트를 실행하기 전에, 아래 값들을 네 환경과 목적에 맞게 수정해줘.

# .env 파일에 저장된 OpenAI API 키의 변수 이름
API_KEY_NAME = "OPENAI_API_KEY"

# 데이터 증강을 실행할 원본 파일 경로
INPUT_FILE = "대출test.csv"

# 증강된 데이터가 저장될 결과 파일 경로
OUTPUT_FILE = "augmented_realtime_final.csv"

# OpenAI 모델 설정 (프롬프트는 함수 내부에 고정)
MODEL_CONFIG = {"model": "gpt-4o-mini", "temperature": 0.9, "max_tokens": 500}

# 각 문장 번역 요청 사이의 대기 시간 (초)
WAIT_TIME_PER_REQUEST = 0.3
# ---------------------------------------------


# OpenAI API 키 불러오기 및 클라이언트 생성
load_dotenv()
openai_api_key = os.getenv(API_KEY_NAME)
if not openai_api_key:
    raise ValueError(f" .env에 {API_KEY_NAME}이 설정되지 않았습니다.")
client = OpenAI(api_key=openai_api_key)


# 2. 한글만 추출하는 후처리 함수
def extract_korean_from_response(text):
    """
    OpenAI 응답에서 한글, 숫자, 기본 구두점만 포함된 문장을 추출합니다.
    """
    # 응답이 비어있으면 그대로 반환
    if not text:
        return ""

    # 여러 줄의 응답 중, 한글이 포함된 내용만 골라냅니다.
    korean_lines = []
    for line in text.split("\n"):
        # 한글이 포함된 줄만 대상으로 합니다.
        if re.search(r"[가-힣]", line):
            # 양쪽의 불필요한 공백이나 따옴표를 먼저 제거합니다.
            clean_line = line.strip().strip("'\"“”")
            korean_lines.append(clean_line)

    # 추출된 한국어 라인들을 하나의 문장으로 합칩니다.
    return " ".join(korean_lines)


# 3. 역번역 함수 (팀과 통일된 프롬프트 사용)
def gpt_back_translate_en(text, client, model_config):
    try:
        # print(f"역번역 시작 (원문 일부): {text[:30]}...") # 너무 많은 로그가 찍히므로 주석 처리
        response = client.chat.completions.create(
            model=model_config["model"],
            messages=[
                {
                    "role": "system",
                    "content": (
                        "너는 한국 보이스피싱 탐지 시스템 개발 프로젝트에 참여 중인 데이터 증강 전문가야. "
                        "사용자가 주는 한국어 문장은 보이스피싱 사례 텍스트야. "
                        "너의 임무는 이 문장을 먼저 영어로 자연스럽게 번역한 뒤, 다시 한국어로 자연스럽고 다양하게 재구성해주는 거야. "
                        "단, 원래 문장의 의미와 맥락은 유지하되 표현을 바꾸고, 자연스럽고 실제 통화처럼 들리도록 만들어야 해. "
                        "**최종 출력은 번역된 한국어 문장만 보여줘.**"
                    ),
                },
                {
                    "role": "user",
                    "content": f"다음 문장을 영어로 번역한 후 다시 한국어로 자연스럽게 재번역해줘. 최종출력은 한국어 문장만:\n\n'{text}'",
                },
            ],
            temperature=model_config["temperature"],
            max_tokens=model_config["max_tokens"],
        )
        result = response.choices[0].message.content.strip()
        # print(f"역번역 완료 (결과 일부): {result[:30]}...") # 너무 많은 로그가 찍히므로 주석 처리
        return result

    except Exception as e:
        print(f"역번역 중 오류 발생: {e}")
        return None  # 오류 발생 시 None 반환


# 4. CSV 파일 불러오기
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"'{INPUT_FILE}' 파일에서 {len(df)}개 행을 불러왔습니다.")
except FileNotFoundError:
    print(f"오류: '{INPUT_FILE}' 파일이 존재하지 않습니다.")
    exit()

# 5. file_name 컬럼 값 변경 (원본이름_trans)
print("\n'file_name' 컬럼 값 변경 작업을 시작합니다...")
df["file_name"] = df["file_name"].astype(str) + "_trans"
print("'file_name' 변경 완료!")


# 6. 실시간 저장을 위한 파일 준비
# 파일이 없으면, 원본 데이터프레임의 헤더를 사용해 새 파일을 생성합니다.
if not os.path.exists(OUTPUT_FILE):
    df.iloc[0:0].to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"결과 파일 '{OUTPUT_FILE}'을 생성하고 헤더를 추가했습니다.")


# 7. 역번역 실행 및 실시간 저장
print("\nOpenAI API로 역번역 및 실시간 저장을 시작합니다...")

# tqdm을 사용하여 전체 진행률 표시
for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    original_text = row["text"]

    # 역번역 함수 호출
    translated_text = gpt_back_translate_en(original_text, client, MODEL_CONFIG)

    # 후처리: 번역 결과가 있고, 비어있지 않은 경우에만 처리
    if translated_text:
        # 후처리 함수로 영어 등 불필요한 내용 제거
        final_text = extract_korean_from_response(translated_text)

        # 만약 한글 추출 후에도 내용이 없다면, 원본을 사용 (안전장치)
        if not final_text:
            final_text = original_text
    else:
        # 번역 실패 시 원본 텍스트 사용
        final_text = original_text

    # 현재 행의 'text' 값을 최종 결과로 업데이트
    row["text"] = final_text

    # 한 줄의 데이터프레임으로 만들어서 파일에 추가(append)
    pd.DataFrame([row]).to_csv(
        OUTPUT_FILE,
        mode="a",
        header=False,
        index=False,
        encoding="utf-8-sig",
        quoting=csv.QUOTE_NONE,
    )

    time.sleep(WAIT_TIME_PER_REQUEST)

print(f"\n모든 작업 완료! 결과가 '{OUTPUT_FILE}' 파일에 저장되었습니다.")

'대출test.csv' 파일에서 6개 행을 불러왔습니다.

'file_name' 컬럼 값 변경 작업을 시작합니다...
'file_name' 변경 완료!
결과 파일 'augmented_realtime_final.csv'을 생성하고 헤더를 추가했습니다.

OpenAI API로 역번역 및 실시간 저장을 시작합니다...


  0%|          | 0/6 [00:00<?, ?it/s]


Error: need to escape, but no escapechar set

In [9]:
# 1. 필요한 라이브러리 불러오기
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm
import time
import re
import csv

# --- (사용자 설정) ---
# 이 스크립트를 실행하기 전에, 아래 값들을 네 환경과 목적에 맞게 수정해줘.

# .env 파일에 저장된 OpenAI API 키의 변수 이름
API_KEY_NAME = "OPENAI_API_KEY"

# 데이터 증강을 실행할 원본 파일 경로
INPUT_FILE = "대출빙자형_역번역.csv"

# 증강된 데이터가 저장될 결과 파일 경로
OUTPUT_FILE = "대출빙자형_trans.csv"

# OpenAI 모델 설정
MODEL_CONFIG = {"model": "gpt-4.1-mini", "temperature": 0.9, "max_tokens": 500}

# 각 대화 처리 후 대기 시간 (초)
WAIT_TIME_PER_FILE = 0.3
# ---------------------------------------------


# OpenAI API 키 불러오기 및 클라이언트 생성
load_dotenv()
openai_api_key = os.getenv(API_KEY_NAME)
if not openai_api_key:
    raise ValueError(f" .env에 {API_KEY_NAME}이 설정되지 않았습니다.")
client = OpenAI(api_key=openai_api_key)


# 한글 문장 추출 함수 (수정된 부분)
def extract_korean(text):
    korean_lines = []
    for line in text.split("\n"):
        if re.search(r"[가-힣]", line):
            clean = line.strip().strip("'\"“”")
            # --- 여기가 핵심 수정 내용 ---
            # 허용 목록에 영어 알파벳(a-zA-Z)을 추가합니다.
            clean = re.sub(r"[^a-zA-Z가-힣0-9\s,\.?!~…·:;\'\"()-]", "", clean)
            # -------------------------
            if clean and len(clean) > 1:
                korean_lines.append(clean)
    return ", ".join(korean_lines)


# 역번역 함수
def gpt_back_translate_en(text, client, model_config):
    try:
        response = client.chat.completions.create(
            model=model_config["model"],
            messages=[
                {
                    "role": "system",
                    "content": (
                        "너는 한국 보이스피싱 탐지 시스템 개발 프로젝트에 참여 중인 데이터 증강 전문가야. "
                        "사용자가 주는 한국어 문장은 보이스피싱 사례 텍스트야. "
                        "너의 임무는 이 문장을 먼저 영어로 자연스럽게 번역한 뒤, 다시 한국어로 자연스럽고 다양하게 재구성해주는 거야. "
                        "단, 원래 문장의 의미와 맥락은 유지하되 표현을 바꾸고, 자연스럽고 실제 통화처럼 들리도록 만들어야 해. "
                        "최종 출력은 번역된 한국어 문장만 보여줘."
                    ),
                },
                {
                    "role": "user",
                    "content": f"다음 문장을 영어로 번역한 후 다시 한국어로 자연스럽게 재번역해줘. 최종출력은 한국어 문장만:\n\n'{text}'",
                },
            ],
            temperature=model_config["temperature"],
            max_tokens=model_config["max_tokens"],
        )
        result = response.choices[0].message.content.strip()
        return extract_korean(result)
    except Exception as e:
        print(f"역번역 중 오류 발생: {e}")
        return None


# 메인 실행 함수
def augment_data(
    df, phishing_types_to_augment, client, save_path, model_config, wait_sec
):
    if not isinstance(phishing_types_to_augment, dict):
        print(
            "오류: phishing_types_to_augment 변수는 {'이름': 숫자} 형태의 딕셔너리여야 합니다."
        )
        return

    if not os.path.exists(save_path):
        pd.DataFrame(columns=["file_name", "phishing_type", "speaker", "text"]).to_csv(
            save_path, index=False, encoding="utf-8-sig"
        )

    for phishing_type, target_count in phishing_types_to_augment.items():
        print(
            f"\n--- '{phishing_type}' 유형 증강 시작 (목표: {target_count}개 대화) ---"
        )
        type_data = df[df["phishing_type"] == phishing_type].copy()

        if type_data.empty:
            print(f"'{phishing_type}' 유형의 데이터가 원본 파일에 없습니다.")
            continue

        files_to_process = type_data["file_name"].unique()

        for file_name in tqdm(files_to_process, desc=f"{phishing_type} 진행률"):
            file_data = type_data[type_data["file_name"] == file_name]
            texts_to_translate = file_data["text"].tolist()

            augmented_texts = [
                gpt_back_translate_en(text, client, model_config)
                for text in texts_to_translate
            ]

            augmented_df = file_data.copy()
            augmented_df["file_name"] = f"{file_name}_trans"

            augmented_df["text"] = [
                aug_text if aug_text and len(aug_text) > 1 else orig_text
                for orig_text, aug_text in zip(texts_to_translate, augmented_texts)
            ]

            augmented_df.to_csv(
                save_path, mode="a", header=False, index=False, encoding="utf-8-sig"
            )

            print(f"처리 완료: {file_name} -> {file_name}_trans")
            time.sleep(wait_sec)


# 메인 코드 실행 부분
if __name__ == "__main__":
    try:
        main_df = pd.read_csv(INPUT_FILE)

        all_phishing_types = {
            phishing_type: main_df[main_df["phishing_type"] == phishing_type][
                "file_name"
            ].nunique()
            for phishing_type in main_df["phishing_type"].unique()
        }
        print("파일의 모든 유형과 대화 개수를 자동으로 설정합니다:")
        print(all_phishing_types)

        augment_data(
            df=main_df,
            phishing_types_to_augment=all_phishing_types,
            client=client,
            save_path=OUTPUT_FILE,
            model_config=MODEL_CONFIG,
            wait_sec=WAIT_TIME_PER_FILE,
        )
        print("\n모든 증강 작업이 완료되었습니다.")

    except FileNotFoundError:
        print(f"오류: 입력 파일({INPUT_FILE})을 찾을 수 없습니다.")
    except Exception as e:
        print(f"스크립트 실행 중 오류가 발생했습니다: {e}")

파일의 모든 유형과 대화 개수를 자동으로 설정합니다:
{'대출빙자형': 432}

--- '대출빙자형' 유형 증강 시작 (목표: 432개 대화) ---


대출빙자형 진행률:   0%|          | 0/432 [00:00<?, ?it/s]

처리 완료: phishing_386 -> phishing_386_trans


대출빙자형 진행률:   0%|          | 1/432 [00:02<16:30,  2.30s/it]

처리 완료: phishing_387 -> phishing_387_trans


...

처리 완료: phishing_389 -> phishing_389_trans


대출빙자형 진행률:   1%|          | 4/432 [00:10<19:03,  2.67s/it]

처리 완료: phishing_390 -> phishing_390_trans


대출빙자형 진행률:   1%|          | 4/432 [00:13<23:31,  3.30s/it]


KeyboardInterrupt: 

In [11]:
# 1. 필요한 라이브러리 불러오기
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm
import time
import re
import csv

# --- (사용자 설정) ---
# 이 스크립트를 실행하기 전에, 아래 값들을 네 환경과 목적에 맞게 수정해줘.

# .env 파일에 저장된 OpenAI API 키의 변수 이름
API_KEY_NAME = "OPENAI_API_KEY"

# 데이터 증강을 실행할 원본 파일 경로
INPUT_FILE = "대출빙자형_역번역.csv"

# 증강된 데이터가 저장될 결과 파일 경로
OUTPUT_FILE = "대출빙자형_trans.csv"

# OpenAI 모델 설정
MODEL_CONFIG = {"model": "gpt-4.1-mini", "temperature": 0.9, "max_tokens": 500}

# 각 대화 처리 후 대기 시간 (초)
WAIT_TIME_PER_FILE = 0.3
# ---------------------------------------------


# OpenAI API 키 불러오기 및 클라이언트 생성
load_dotenv()
openai_api_key = os.getenv(API_KEY_NAME)
if not openai_api_key:
    raise ValueError(f" .env에 {API_KEY_NAME}이 설정되지 않았습니다.")
client = OpenAI(api_key=openai_api_key)


# 한글 문장 추출 함수
def extract_korean(text):
    korean_lines = []
    for line in text.split("\n"):
        if re.search(r"[가-힣]", line):
            clean = line.strip().strip("'\"“”")
            clean = re.sub(r"[^a-zA-Z가-힣0-9\s,\.?!~…·:;\'\"()-]", "", clean)
            if clean and len(clean) > 1:
                korean_lines.append(clean)
    return ", ".join(korean_lines)


# 역번역 함수
def gpt_back_translate_en(text, client, model_config):
    try:
        response = client.chat.completions.create(
            model=model_config["model"],
            messages=[
                {
                    "role": "system",
                    "content": (
                        "너는 한국 보이스피싱 탐지 시스템 개발 프로젝트에 참여 중인 데이터 증강 전문가야. "
                        "사용자가 주는 한국어 문장은 보이스피싱 사례 텍스트야. "
                        "너의 임무는 이 문장을 먼저 영어로 자연스럽게 번역한 뒤, 다시 한국어로 자연스럽고 다양하게 재구성해주는 거야. "
                        "단, 원래 문장의 의미와 맥락은 유지하되 표현을 바꾸고, 자연스럽고 실제 통화처럼 들리도록 만들어야 해. "
                        "**최종 출력은 번역된 한국어 문장만 보여줘.**"
                        "출력에 영어 단어가 섞이거나 설명이 포함되면 실패야"
                    ),
                },
                {
                    "role": "user",
                    "content": f"다음 문장을 영어로 번역한 뒤, 다시 자연스럽고 실제 대화처럼 들리게 한국어로 재번역해줘. "
                        f"출력에는 반드시 한국어 문장만 포함돼야 해. 영어 단어, 설명, 번역 과정은 절대 포함하지 마:\n\n'{text}'",
                },
            ],
            temperature=model_config["temperature"],
            max_tokens=model_config["max_tokens"],
        )
        result = response.choices[0].message.content.strip()
        return extract_korean(result)
    except Exception as e:
        print(f"역번역 중 오류 발생: {e}")
        return None


# --- (수정 필요 1: 이미 처리된 파일 목록을 불러오는 함수 추가) ---
def load_processed_files(save_path):
    """
    결과 파일(save_path)을 읽어서, 이미 처리된 'file_name' 목록을 반환합니다.
    """
    if not os.path.exists(save_path):
        return set()  # 파일이 없으면 빈 세트를 반환
    try:
        # 파일이 비어있는 경우를 대비하여 예외 처리
        if os.path.getsize(save_path) > 0:
            processed_df = pd.read_csv(save_path)
            # '_trans'가 붙은 파일 이름들을 집합(set)으로 만들어 반환합니다.
            return set(processed_df["file_name"].unique())
        else:
            return set()
    except (pd.errors.EmptyDataError, KeyError):
        # 파일은 있지만 비어있거나, 컬럼이 없는 경우
        return set()


# -------------------------------------------------------------


# 메인 실행 함수
def augment_data(
    df,
    phishing_types_to_augment,
    client,
    save_path,
    model_config,
    wait_sec,
    processed_files,
):
    if not isinstance(phishing_types_to_augment, dict):
        print(
            "오류: phishing_types_to_augment 변수는 {'이름': 숫자} 형태의 딕셔너리여야 합니다."
        )
        return

    if not os.path.exists(save_path):
        pd.DataFrame(columns=["file_name", "phishing_type", "speaker", "text"]).to_csv(
            save_path, index=False, encoding="utf-8-sig"
        )

    for phishing_type, target_count in phishing_types_to_augment.items():
        print(
            f"\n--- '{phishing_type}' 유형 증강 시작 (목표: {target_count}개 대화) ---"
        )
        type_data = df[df["phishing_type"] == phishing_type].copy()

        if type_data.empty:
            print(f"'{phishing_type}' 유형의 데이터가 원본 파일에 없습니다.")
            continue

        files_to_process = type_data["file_name"].unique()

        for file_name in tqdm(files_to_process, desc=f"{phishing_type} 진행률"):
            # --- (수정 필요 2: 이미 처리된 파일인지 확인하고 건너뛰는 로직) ---
            new_file_name = f"{file_name}_trans"
            if new_file_name in processed_files:
                print(f"이미 처리됨: {new_file_name}, 건너뜁니다.")
                continue
            # -----------------------------------------------------------------

            file_data = type_data[type_data["file_name"] == file_name]
            texts_to_translate = file_data["text"].tolist()

            augmented_texts = [
                gpt_back_translate_en(text, client, model_config)
                for text in texts_to_translate
            ]

            augmented_df = file_data.copy()
            augmented_df["file_name"] = new_file_name  # 위에서 만든 새 파일 이름 사용

            augmented_df["text"] = [
                aug_text if aug_text and len(aug_text) > 1 else orig_text
                for orig_text, aug_text in zip(texts_to_translate, augmented_texts)
            ]

            augmented_df.to_csv(
                save_path, mode="a", header=False, index=False, encoding="utf-8-sig"
            )

            print(f"처리 완료: {file_name} -> {new_file_name}")
            time.sleep(wait_sec)


# 메인 코드 실행 부분
if __name__ == "__main__":
    try:
        main_df = pd.read_csv(INPUT_FILE)

        # 스크립트 시작 시, 이미 처리된 파일 목록을 불러옵니다.
        processed_files = load_processed_files(OUTPUT_FILE)
        print(f"이미 처리된 대화 {len(processed_files)}개를 확인했습니다.")

        all_phishing_types = {
            phishing_type: main_df[main_df["phishing_type"] == phishing_type][
                "file_name"
            ].nunique()
            for phishing_type in main_df["phishing_type"].unique()
        }
        print("파일의 모든 유형과 대화 개수를 자동으로 설정합니다:")
        print(all_phishing_types)

        augment_data(
            df=main_df,
            phishing_types_to_augment=all_phishing_types,
            client=client,
            save_path=OUTPUT_FILE,
            model_config=MODEL_CONFIG,
            wait_sec=WAIT_TIME_PER_FILE,
            processed_files=processed_files,  # 처리 목록을 함수에 전달
        )
        print("\n모든 증강 작업이 완료되었습니다.")

    except FileNotFoundError:
        print(f"오류: 입력 파일({INPUT_FILE})을 찾을 수 없습니다.")
    except Exception as e:
        print(f"스크립트 실행 중 오류가 발생했습니다: {e}")

이미 처리된 대화 240개를 확인했습니다.
파일의 모든 유형과 대화 개수를 자동으로 설정합니다:
{'대출빙자형': 432}

--- '대출빙자형' 유형 증강 시작 (목표: 432개 대화) ---


대출빙자형 진행률:   0%|          | 0/432 [00:00<?, ?it/s]

이미 처리됨: phishing_386_trans, 건너뜁니다.
이미 처리됨: phishing_387_trans, 건너뜁니다.
이미 처리됨: phishing_388_trans, 건너뜁니다.
이미 처리됨: phishing_389_trans, 건너뜁니다.
이미 처리됨: phishing_390_trans, 건너뜁니다.
이미 처리됨: phishing_391_trans, 건너뜁니다.
이미 처리됨: phishing_392_trans, 건너뜁니다.
이미 처리됨: phishing_393_trans, 건너뜁니다.
이미 처리됨: phishing_394_trans, 건너뜁니다.
이미 처리됨: phishing_395_trans, 건너뜁니다.
이미 처리됨: phishing_396_trans, 건너뜁니다.
이미 처리됨: phishing_397_trans, 건너뜁니다.
이미 처리됨: phishing_398_trans, 건너뜁니다.
이미 처리됨: phishing_399_trans, 건너뜁니다.
이미 처리됨: phishing_400_trans, 건너뜁니다.
이미 처리됨: phishing_401_trans, 건너뜁니다.
이미 처리됨: phishing_402_trans, 건너뜁니다.
이미 처리됨: phishing_403_trans, 건너뜁니다.
이미 처리됨: phishing_404_trans, 건너뜁니다.
이미 처리됨: phishing_405_trans, 건너뜁니다.
이미 처리됨: phishing_406_trans, 건너뜁니다.
이미 처리됨: phishing_407_trans, 건너뜁니다.
이미 처리됨: phishing_408_trans, 건너뜁니다.
이미 처리됨: phishing_409_trans, 건너뜁니다.
이미 처리됨: phishing_410_trans, 건너뜁니다.
이미 처리됨: phishing_411_trans, 건너뜁니다.
이미 처리됨: phishing_412_trans, 건너뜁니다.
이미 처리됨: phishing_413_trans, 건너뜁니다.
이미 처리됨: phishing_414

대출빙자형 진행률:  56%|█████▌    | 241/432 [00:39<00:31,  6.14it/s]

처리 완료: phishing_1441 -> phishing_1441_trans


...

처리 완료: phishing_2045 -> phishing_2045_trans


대출빙자형 진행률: 100%|█████████▉| 431/432 [17:11:33<00:23, 23.52s/it]

처리 완료: phishing_2046 -> phishing_2046_trans


대출빙자형 진행률: 100%|██████████| 432/432 [17:11:36<00:00, 143.28s/it]


모든 증강 작업이 완료되었습니다.


In [17]:
import pandas as pd
import numpy as np

# --- 설정 ---
FILE_PATH = "대출빙자형_역번역.csv"
OUTPUT_FILENAME = "stratified_samples_150_files.csv"

# 샘플링할 총 file_name 개수
TOTAL_FILES_TO_SAMPLE = 150

# --- 1단계: EDA 및 대화 길이 측정 ---
print("--- 1단계: EDA 및 대화 길이 측정을 시작합니다. ---")
try:
    # 사용자가 알려준 컬럼명을 사용하여 CSV 파일을 읽어옵니다.
    df = pd.read_csv(FILE_PATH)

    # 제공된 컬럼명이 파일에 모두 있는지 확인합니다.
    required_columns = ["file_name", "phishing_type", "speaker", "text"]
    if not all(col in df.columns for col in required_columns):
        print("[오류] 파일에 필요한 컬럼이 모두 존재하지 않습니다.")
        print(f"필요한 컬럼: {required_columns}")
        print(f"파일에 있는 컬럼: {df.columns.tolist()}")
        exit()

    # 단어 수 계산을 위해 text 컬럼의 결측치를 빈 문자열로 처리합니다.
    df["text"] = df["text"].fillna("")
    df["word_count"] = df["text"].str.split().str.len()

    # file_name을 기준으로 그룹화하여 대화별 총 단어 수를 계산합니다.
    conversation_summary = (
        df.groupby("file_name")
        .agg(total_word_count=("word_count", "sum"))
        .reset_index()
    )

    print("\n각 대화(file_name)별 총 단어 수 계산 완료.")
    print(conversation_summary.head())

except FileNotFoundError:
    print(f"[오류] '{FILE_PATH}' 파일을 찾을 수 없습니다.")
    exit()
except Exception as e:
    print(f"[오류] 데이터를 읽는 중 문제가 발생했습니다: {e}")
    exit()

# --- 2단계: 대화 길이 기준 계층화 (Stratification) ---
print("\n--- 2단계: 대화 길이를 기준으로 그룹화(계층화)합니다. ---")

# qcut을 사용하여 총 단어 수를 기준으로 3개의 그룹(짧음, 중간, 김)으로 나눕니다.
# duplicates='drop' 옵션으로 동일한 경계값이 있어도 에러 없이 처리합니다.
try:
    conversation_summary["length_group"] = pd.qcut(
        conversation_summary["total_word_count"],
        q=3,
        labels=["short", "medium", "long"],
        duplicates="drop",
    )
except ValueError:
    # 데이터가 너무 적거나 분포가 특이하여 3개 그룹으로 나눌 수 없을 경우 2개로 시도
    conversation_summary["length_group"] = pd.qcut(
        conversation_summary["total_word_count"],
        q=2,
        labels=["short", "long"],
        duplicates="drop",
    )


print("\n[대화 길이 그룹별 파일 개수]")
group_counts = conversation_summary["length_group"].value_counts()
print(group_counts)

# --- 3단계: 그룹 비율에 맞춰 비례 샘플링 ---
print("\n--- 3단계: 그룹 비율에 맞춰 150개 파일을 샘플링합니다. ---")

# 각 그룹에서 샘플링할 파일 개수를 비율에 맞게 계산합니다.
# 소수점 계산 후 반올림하고, 전체 합이 150이 되도록 조정합니다.
n_total_files = len(conversation_summary)
sample_counts = (
    (group_counts / n_total_files * TOTAL_FILES_TO_SAMPLE).round().astype(int)
)

# 반올림 오차로 합계가 150이 안 될 경우, 가장 큰 그룹에서 개수를 조정합니다.
diff = TOTAL_FILES_TO_SAMPLE - sample_counts.sum()
if diff != 0:
    sample_counts[sample_counts.idxmax()] += diff

print("\n[그룹별 샘플링할 파일 개수]")
print(sample_counts)

# 계층적 샘플링 실행
sampled_files_list = []
for group_name, n_sample in sample_counts.items():
    # 해당 그룹에 속하는 file_name들 중에서 n_sample 개수만큼 무작위로 추출
    files_in_group = conversation_summary[
        conversation_summary["length_group"] == group_name
    ]
    # 만약 그룹의 파일 수가 샘플링할 개수보다 적으면 있는 만큼만 샘플링
    n_sample = min(n_sample, len(files_in_group))
    sampled_files = files_in_group.sample(n=n_sample, random_state=42)
    sampled_files_list.append(sampled_files)

# 샘플링된 file_name들을 하나로 합치고, 그 이름들만 리스트로 추출
final_sampled_summary = pd.concat(sampled_files_list)
sampled_file_names = final_sampled_summary["file_name"].tolist()

print(f"\n총 {len(sampled_file_names)}개의 file_name 샘플링 완료.")

# --- 4단계: 최종 데이터 추출 및 저장 ---
print("\n--- 4단계: 샘플링된 파일들의 전체 데이터를 저장합니다. ---")

# 원본 데이터(df)에서 샘플링된 file_name에 해당하는 모든 행을 필터링합니다.
final_df = df[df["file_name"].isin(sampled_file_names)].copy()

# 분석에 사용했던 'word_count' 컬럼은 최종 파일에서 제외합니다.
final_df = final_df.drop(columns=["word_count"])

# 최종 결과를 CSV 파일로 저장합니다.
final_df.to_csv(OUTPUT_FILENAME, index=False, encoding="utf-8-sig")

print(f"\n성공: '{OUTPUT_FILENAME}' 파일 저장이 완료되었습니다.")
print(
    f"이 파일에는 {len(sampled_file_names)}개의 고유한 대화(file_name)에 해당하는 총 {len(final_df)}개의 행이 포함되어 있습니다."
)

--- 1단계: EDA 및 대화 길이 측정을 시작합니다. ---

각 대화(file_name)별 총 단어 수 계산 완료.
       file_name  total_word_count
0  phishing_1026                65
1  phishing_1027                49
2  phishing_1028                62
3  phishing_1029                64
4  phishing_1030                59

--- 2단계: 대화 길이를 기준으로 그룹화(계층화)합니다. ---

[대화 길이 그룹별 파일 개수]
length_group
short     145
long      144
medium    143
Name: count, dtype: int64

--- 3단계: 그룹 비율에 맞춰 150개 파일을 샘플링합니다. ---

[그룹별 샘플링할 파일 개수]
length_group
short     50
long      50
medium    50
Name: count, dtype: int64

총 150개의 file_name 샘플링 완료.

--- 4단계: 샘플링된 파일들의 전체 데이터를 저장합니다. ---

성공: 'stratified_samples_150_files.csv' 파일 저장이 완료되었습니다.
이 파일에는 150개의 고유한 대화(file_name)에 해당하는 총 3470개의 행이 포함되어 있습니다.
